# NumPy Sort, Search & Filter

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 6/7

Ranking results, finding where conditions hold, counting categories — these operations power everything from leaderboards to top-k recommendations. This lesson collects NumPy's whole toolbox for them.

## 🎯 Learning Objectives

- Sort arrays safely: `np.sort` (copy) versus `arr.sort()` (in place)
- Sort 2-D arrays along either axis
- Rank data with `argsort` and extract the top-k pattern
- Use all three forms of `np.where`: indices, elementwise if/else, nested branches
- Locate values with `nonzero` / `count_nonzero` and summarize with `np.unique`
- Clamp ranges with `np.clip` and find insertion points with `searchsorted`

## 1. np.sort(arr) vs arr.sort(): Copy vs In-Place

Two spellings, two behaviors. `np.sort(arr)` returns a NEW sorted array and leaves the original alone. `arr.sort()` reorders the array ITSELF and returns None — the original order is gone.

**Syntax:**
```python
ordered = np.sort(arr)   # sorted COPY, source untouched
arr.sort()               # sorts IN PLACE, returns None
```

In [ ]:
import numpy as np

times = np.array([14.2, 11.9, 13.5, 10.8])   # race times

ordered = np.sort(times)         # safe: new array
print("sorted copy:", ordered)
print("original   :", times, "- still in finish-line order")

times.sort()                     # destructive: mutates times forever
print("after .sort():", times)

In [ ]:
import numpy as np

times = np.array([14.2, 11.9, 13.5, 10.8])

print("descending via reverse :", np.sort(times)[::-1])
print("descending via negation:", -np.sort(-times))   # negate, sort, negate back

## 2. Sorting a 2-D Array Along an Axis

With `axis=` you sort each row or each column independently. Remember the axis rule from last lesson: the named axis is the direction of travel.

**Syntax:**
```python
np.sort(m, axis=1)   # each row sorted left to right (the default)
np.sort(m, axis=0)   # each column sorted top to bottom
```

In [ ]:
import numpy as np

m = np.array([[9, 3, 7],
              [4, 8, 1],
              [6, 5, 2]])

print("sort each ROW (axis=1):")
print(np.sort(m, axis=1))
print()
print("sort each COLUMN (axis=0):")
print(np.sort(m, axis=0))

## 3. argsort: The Indices That Would Sort

Often you need not just sorted VALUES but the ORDER — so you can reorder names alongside scores. `argsort` returns the positions that would sort the array; index back to get sorted data or parallel arrays.

**Syntax:**
```python
order = np.argsort(x)     # indices that sort x ascending
x[order]                  # equals np.sort(x)
names[order]              # labels reordered the same way

top_k = np.argsort(x)[-k:][::-1]   # positions of the k largest, best first
```

In [ ]:
import numpy as np

names = np.array(["Rafi", "Nadia", "Omar", "Priya"])
points = np.array([72, 91, 65, 88])

order = np.argsort(points)             # who comes 1st, 2nd, 3rd...
print("ranking order :", order)
print("sorted points :", points[order])
print("sorted names  :", names[order])

In [ ]:
import numpy as np

scores = np.array([78, 92, 61, 88, 95, 73])

idx_asc = np.argsort(scores)[-3:]      # positions of the 3 biggest (smallest first)
idx_top = idx_asc[::-1]                # flip -> best score first
print("top-3 indices:", idx_top)
print("their scores :", scores[idx_top])

> 🔍 **Under the Hood:** `np.sort` defaults to introsort — quicksort with fallbacks — running in O(n log n), and it is NOT stable (equal keys may swap places). Pass `kind="stable"` when tie order matters. `argsort` carries the indices along, so it costs roughly twice the data movement. When you only need the k smallest or largest, `np.argpartition(x, k)` does O(n) partial work instead of fully sorting — the trick behind fast top-k retrieval over millions of items.

## 4. np.where(cond): Find Where Conditions Hold

Form 1 — pass only the condition and get the INDICES of every True position. Note it returns a TUPLE (one index array per dimension), even for 1-D input.

**Syntax:**
```python
np.where(cond)        # tuple of index arrays
idx = np.where(x > 5)[0]   # unwrap for 1-D use
```

In [ ]:
import numpy as np

temps = np.array([31.5, 33.8, 30.2, 34.1, 32.6])

found = np.where(temps > 32)
print("raw result      :", found, "(a tuple!)")
indices = found[0]
print("hot day indices :", indices)
print("hot temperatures:", temps[indices])

## 5. np.where(cond, a, b): Element-wise If/Else

Form 2 is a vectorized ternary: choose `a` where the condition is True and `b` where it is False — decided per element, no loop needed.

**Syntax:**
```python
np.where(cond, value_if_true, value_if_false)
np.where(x < 0, 0, x)    # clamp negatives to zero
```

In [ ]:
import numpy as np

scores = np.array([45, 82, 67, 91, 58])

verdicts = np.where(scores >= 60, "PASS", "FAIL")
for score, verdict in zip(scores.tolist(), verdicts.tolist()):
    print(score, "->", verdict)

signal = np.array([3, -1, 7, -5, 2])
print("negatives clamped:", np.where(signal < 0, 0, signal))

## 6. Nested where: Multi-way Branching

Need more than two outcomes? Nest `where` calls — each level handles one more branch, exactly like chained if/elif/else.

**Syntax:**
```python
np.where(c1, "A",
         np.where(c2, "B",
                  "C"))
```

In [ ]:
import numpy as np

scores = np.array([45, 82, 67, 91, 58])

grades = np.where(scores >= 80, "A",
                  np.where(scores >= 60, "B", "C"))
for score, grade in zip(scores.tolist(), grades.tolist()):
    print(score, "->", grade)

## 7. nonzero and count_nonzero

`np.nonzero` reports every position holding a non-zero value — the classic tool for sparse data. `count_nonzero` just tallies them, and since `True == 1`, it doubles as a mask counter.

**Syntax:**
```python
np.nonzero(arr)          # tuple of index arrays of non-zeros
np.count_nonzero(arr)    # how many non-zeros (or Trues)
```

In [ ]:
import numpy as np

sparse = np.array([0, 4, 0, 9, 0, 0, 2])

positions = np.nonzero(sparse)[0]
print("non-zero indices:", positions)
print("their values    :", sparse[positions])
print("how many        :", np.count_nonzero(sparse))

mask = np.array([True, False, True, True])
print("True count      :", np.count_nonzero(mask))   # Trues count as 1

## 8. np.unique: Categories and Counts

`np.unique` returns the sorted distinct values — and with `return_counts=True`, how often each occurs. Two lines give you a full frequency table, no pandas required.

**Syntax:**
```python
values = np.unique(x)                       # sorted unique values
values, counts = np.unique(x, return_counts=True)
```

In [ ]:
import numpy as np

votes = np.array([2, 3, 2, 1, 3, 2, 4, 1, 2])   # votes for candidates 1-4

candidates, counts = np.unique(votes, return_counts=True)
for c, n in zip(candidates, counts):
    print(f"candidate {c}: {n} votes")

winner = candidates[counts.argmax()]
print("winner: candidate", winner)

## 9. np.clip: Force Values Into Range

`clip` pushes everything below a floor up to it, and everything above a ceiling down to it. One call replaces a whole if/else cleaning routine — pass None to skip one side.

**Syntax:**
```python
np.clip(arr, low, high)
np.clip(arr, 0, None)    # only enforce the lower bound
```

In [ ]:
import numpy as np

readings = np.array([-12, 5, 42, 87, 103, 60])   # sensor spikes and dropouts

print("clamped to [0, 100]:", np.clip(readings, 0, 100))
print("only lower bound   :", np.clip(readings, 0, None))

## 10. searchsorted: Where Does This Belong?

Given a SORTED array, `searchsorted` tells you the index at which a value could be inserted while keeping order — in O(log n), no scanning.

**Syntax:**
```python
np.searchsorted(sorted_arr, value)                # leftmost slot
np.searchsorted(sorted_arr, value, side="right")  # after equal values
```

In [ ]:
import numpy as np

queue = np.array([10, 20, 30, 40, 50])   # already sorted

pos = np.searchsorted(queue, 35)
print("35 belongs at index", pos)

queue = np.insert(queue, pos, 35)        # actually insert it
print(queue)

dups = np.array([10, 20, 20, 30])
print("side='left' ->", np.searchsorted(dups, 20), "| side='right' ->", np.searchsorted(dups, 20, side="right"))

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Confusing `np.sort(a)` with `a.sort()` | The latter destroys the original order permanently | Use `np.sort(a)` unless mutation is intended |
| Treating `argsort` output as values | It returns INDICES — printing them looks like nonsense | Index back: `x[np.argsort(x)]` |
| Unwrapping `np.where(cond)` carelessly | Form 1 returns a TUPLE of arrays | Take `[0]` for 1-D: `np.where(cond)[0]` |
| Sorting descending by `-x` on unsigned dtypes | Negating `uint8/uint16` wraps around | Cast to float/int first, or use `[::-1]` |
| Calling `searchsorted` on unsorted data | Result is silently meaningless | Sort first — the contract requires sorted input |

## 💡 Best Practices & Pro Tips

- Reach for `kind="stable"` when ties must keep their original relative order.
- On large arrays, prefer `np.argpartition(x, k)[:k]` over full `argsort` for pure top-k needs.
- `np.unique(..., return_counts=True)` is your instant value_counts — great for quick category audits.
- `np.clip` is standard hygiene on model outputs: keep probabilities in `[0, 1]`, pixel values in `[0, 255]`.
- **AI-engineering relevance:** top-k via argsort drives recommendations, beam search and classification confidences; `np.where` builds targets, masks and cleaned columns; `clip` guards against exploding logits.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `np.sort(a)` | Sorted copy | `np.sort(times)` |
| `a.sort()` | In-place sort (destroys order) | `times.sort()` |
| `np.sort(m, axis=k)` | Per-row / per-column sort | `np.sort(m, axis=0)` |
| `np.argsort(x)` | Indices that would sort | `x[np.argsort(x)]` |
| `np.argsort(x)[-k:][::-1]` | Top-k indices, best first | leaderboard pattern |
| `np.where(cond)` | Indices where True (tuple!) | `np.where(t > 32)[0]` |
| `np.where(c, a, b)` | Element-wise if/else | `np.where(s >= 60, "PASS", "FAIL")` |
| `np.nonzero(x)` / `count_nonzero` | Non-zero positions / tally | `np.count_nonzero(mask)` |
| `np.unique(x, return_counts=True)` | Distinct values + frequencies | frequency table |
| `np.clip(x, lo, hi)` | Force into range | `np.clip(r, 0, 100)` |
| `np.searchsorted(sorted_x, v)` | Insertion point keeping order | `searchsorted(q, 35)` |

Key takeaways:
- Copy (`np.sort`) vs in-place (`.sort()`) is a recurring NumPy theme — know which you are calling.
- `argsort` converts sorting into reordering ANY parallel data.
- `np.where` has three faces: find indices, choose between two values, nest for multi-way logic.
- `unique + return_counts` answers "what categories exist and how common are they?" in one line.

## 🔗 Next Lesson

- Continue to **[07_Random_Numbers](../07_Random_Numbers/notes.ipynb)** — the modern Generator API, reproducible seeds, and your first simulations.